In [1]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
from Bio import SeqIO

site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
test_fasta = "../../general_input/test/testsuperset.fasta"
embeddings_path = "../input/test_embeddings.npy"

In [ ]:
records = SeqIO.parse(test_fasta, "fasta")
sequences = []

for record in records:
    sequences.append(str(record.seq))
    
print(f"Total sequences: {len(sequences)}")

Total sequences: 224309


In [6]:
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16).to(device)
model.eval()

In [8]:
def get_embeddings_low_vram(sequences, batch_size=2):
    embeddings = []
    
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i : i + batch_size]
        
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=1024)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            
        batch_embeddings = outputs.last_hidden_state.mean(dim=1).float()
        
        embeddings.append(batch_embeddings.cpu().numpy())
        
    return np.concatenate(embeddings, axis=0)

In [ ]:
try:
    embeddings = get_embeddings_low_vram(sequences, batch_size=2)
    print(f"Embeddings generated. Shape: {embeddings.shape}")
    np.save(embeddings_path, embeddings)
    print("Saved embeddings to disk.")
except RuntimeError as e:
    if "out of memory" in str(e):
        print("MemoryError: Not enough GPU memory to process the batch size. Try reducing the batch size.")
    else:
        print(f"An unexpected error ocurred: {e}")

100%|██████████| 112155/112155 [11:45:50<00:00,  2.65it/s]  


Embeddings generated. Shape: (224309, 1280)
Saved embeddings to disk.
